In [13]:
# import sys
# !{sys.executable} -m pip install opentrons
#!pip install opentrons

In [14]:
## Imports
from opentrons import protocol_api, execute, simulate
from opentrons.types import Location, Point
import pandas as pd
import numpy as np
#from data import data_colordf

# Call protocol to simulate or Execute

In [15]:
protocol = simulate.get_protocol_api('2.10')
#protocol = execute.get_protocol_api('2.15')

/Users/clarat/.opentrons/robot_settings.json not found. Loading defaults
Deck calibration not found.


/Users/clarat/.opentrons/deck_calibration.json not found. Loading defaults


# Define labwear

In [16]:
## pipette and labware
tiprack = protocol.load_labware('opentrons_96_tiprack_300ul', '8')
pip = protocol.load_instrument('p300_single_gen2', 'right', tip_racks=[tiprack])
stock = protocol.load_labware('corning_6_wellplate_16.8ml_flat',location='1')
plate = protocol.load_labware('opentrons_96_wellplate_200ul_pcr_full_skirt','2')
#flat_plate = protocol.load_labware('nest_96_wellplate_200ul_flat','2')

# Define locations of stock solution

In [17]:
start_stock = {
    'magenta_loc': [stock.wells()[0].center(),stock.wells()[1].center()],#stock.wells()[0].center()
    #'water_loc': loc_stock(3,stock.wells()[0].center())+loc_stock(3,stock.wells()[1].center()),#stock.wells()[0].center()
    'magenta_vol':[20000,20000] ,# adjust the volumes for each(6) concentration
    'cyan_loc':[stock.wells()[2].center(),stock.wells()[3].center()], #stock.wells()[1].center()
    #'dmf_loc':loc_stock(3,stock.wells()[2].center())+loc_stock(3,stock.wells()[3].center()), #stock.wells()[1].center()
    'cyan_vol':[20000,20000],# adjust the volumes for each(6) concentration
    'yellow_loc':[stock.wells()[4].center(),stock.wells()[5].center()],#stock.wells()[2].center()
    #'ethanol_loc':loc_stock(3,stock.wells()[4].center())+loc_stock(3,stock.wells()[5].center()),#stock.wells()[2].center()
    'yellow_vol':[20000,20000]# adjust the volumes for each(6) concentration
}
start_stock = pd.DataFrame(data=start_stock)


# Relevant Functions

In [18]:
def loc_stock(n_samples,fill):
    array = []
    #fill array
    for i in range(n_samples): 
        array.append(fill) 
    return array

def loc_mixing(start,n_sample,plate): 
    array = []
    for i in range(n_sample):
        array.append(plate.wells()[i+start].center())
    return array

In [19]:
# funtion for moving liquid since repetative
def pipette_action(protocol, pipette, stock_volume_to_pull, stock_position_to_pull, destination_well,stockname):
    pipette.aspirate(stock_volume_to_pull, stock_position_to_pull)
    pipette.dispense(stock_volume_to_pull, destination_well)
    pipette.touch_tip()  
    #pipette.blow_out(destination_well)

In [20]:
def stock_in_all(stock,stock_location,tip_max_vol,dis_tray,name):
    '''
    Args- 
    stock: 1D array of all the stock volumes
    stock_location: 1D array of the stock locations corresponding to volumes
    tip_max_vol: float num of the max volume tip can hold
    dis_tray: 1D array of all the locaitons for dispensing
    name: stock name str

    Function-
    Determine the number of times needed to get stock and dispence. 
    If the volume is larger than the tip can hold, automatically determined how many times it has to aspirate and dispence. 
    Care since stock, stock_location, dis_tray are all arrays
    '''
    for i in range(len(stock)):
        stock_amount = stock[i]
        if stock_amount == 0:
            pass
        elif stock[i] >= tip_max_vol: # eg:300 is max tip volume
            n = stock[i]//tip_max_vol
            if n > 0: 
                for count in range(int(n)):
                    pipette_action(protocol, 
                            pipette = pip, 
                            stock_volume_to_pull = tip_max_vol, 
                            stock_position_to_pull = stock_location[i], 
                            destination_well = dis_tray[i],
                            stockname= name )
                    stock_amount -= tip_max_vol
                    
                    if stock_amount <= stock[i] - tip_max_vol*n and stock_amount != 0:
                        pipette_action(protocol, 
                                    pipette = pip, 
                                    stock_volume_to_pull = (stock[i] - tip_max_vol*n),
                                    stock_position_to_pull = stock_location[i], 
                                    destination_well = dis_tray[i],
                                    stockname= name )
        else: 
            pipette_action(protocol, 
                           pipette = pip, 
                           stock_volume_to_pull = stock[i], 
                           stock_position_to_pull = stock_location[i], 
                           destination_well = dis_tray[i],
                           stockname= name )
        

In [21]:
def loc_stock(n_samples,fill):
    array = []
    #fill array
    for i in range(n_samples): 
        array.append(fill) 
    return array

def loc_mixing(start,n_sample): 
    array = []
    for i in range(n_sample):
        array.append(plate.wells()[i+start].center())
    return array

#let = 'D'
#RGB_stocksandlocations = {'mix_loc': [plate[str(let)+'1'].center(), plate[str(let)+'2'].center(),plate[str(let)+'3'].center(),plate[str(let)+'4'].center(),plate[str(let)+'5'].center(),plate[str(let)+'6'].center(),plate[str(let)+'7'].center(),plate[str(let)+'8'].center(),plate[str(let)+'9'].center(),plate[str(let)+'10'].center(),plate[str(let)+'11'].center(),plate[str(let)+'12'].center()], 

In [22]:
##Calling the function
N_samples = 96

loc_red_stoc = loc_stock(N_samples,start_stock['magenta_loc'][0])
loc_blue_stoc = loc_stock(N_samples,start_stock['cyan_loc'][0])
loc_yellow_stoc = loc_stock(N_samples,start_stock['yellow_loc'][0])

loc_mixing = loc_mixing(0,N_samples) #Start is 0 - 95, A: 0-11, B: 12-23, 
loc_mixing

[Location(point=Point(x=146.88, y=74.24, z=8.525), labware=A1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2),
 Location(point=Point(x=146.88, y=65.24, z=8.525), labware=B1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2),
 Location(point=Point(x=146.88, y=56.24, z=8.525), labware=C1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2),
 Location(point=Point(x=146.88, y=47.24, z=8.525), labware=D1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2),
 Location(point=Point(x=146.88, y=38.24, z=8.525), labware=E1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2),
 Location(point=Point(x=146.88, y=29.24, z=8.525), labware=F1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2),
 Location(point=Point(x=146.88, y=20.24, z=8.525), labware=G1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2),
 Location(point=Point(x=146.88, y=11.24, z=8.525), labware=H1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2),
 Locatio

# Volumes and destination locations

In [23]:

RGB_stocksandlocations = {
    'mix_loc': loc_mixing,
## Clara BO iteration: 
'stock_red_vol': [ 40., 130.,  67.,   3.,  43., 142., 118.,  59.,  78.,  38.,  87.,
        139., 135.,  82.,  88.,  76., 150.,  59., 203., 151., 121., 114.,
        177., 110., 140.,  51.,  82.,  73., 164., 174., 109., 119.,  82.,
         96.,   8.,  64., 125., 166.,  83.,  60., 150., 191., 148.,  84.,
         95., 105., 148., 113., 123., 104.,   1.,  89.,  30.,  17., 128.,
         10., 128.,  41.,  72.,  75., 117., 141.,  40.,  57., 125.,  34.,
         25.,  64.,  63., 217.,  86.,  91., 156.,  86., 131.,  79., 125.,
         92., 119.,  44.,  57.,  19., 190., 134., 129.,  52.,  70., 240.,
         25., 162.,  21., 159., 237., 120.,  53., 134.],
'stock_blue_vol':[103., 103.,  74., 108., 129.,  74., 140., 131., 110., 151., 139.,
         20.,  71., 161.,  96., 112., 140., 122.,  49., 117.,  58., 113.,
         53., 135.,  16., 229., 130.,  13.,  42.,  64., 143.,  75.,  37.,
         20., 104., 159.,  44.,  25., 146.,  65.,  44., 105.,  35.,  98.,
         56.,  79.,  55.,  80., 118., 126., 231., 106., 239.,  94., 107.,
        113., 108., 144.,  80.,  71., 105.,  10., 213., 203., 166., 133.,
        146., 124., 138.,  59.,   2.,  81., 131.,  81., 114.,  27., 104.,
          7., 135.,  82., 134., 122.,   4., 113.,  40., 123., 127.,  39.,
         58., 116., 152.,  95.,  52., 134., 110., 131.],
'stock_yellow_vol':[156.,  65., 158., 187., 127.,  83.,  40., 109., 110., 110.,  73.,
        140.,  92.,  56., 114., 110.,   9., 118.,  46.,  31., 119.,  72.,
         69.,  54., 143.,  19.,  87., 213.,  92.,  60.,  46., 104., 179.,
        183., 186.,  75., 129., 108.,  69., 174., 104.,   3., 115., 116.,
        148., 114.,  96., 105.,  58.,  68.,  67., 103.,  29., 187.,  64.,
        176.,  63., 114., 146., 152.,  76., 147.,  45.,  39.,   7., 132.,
        127., 111.,  97.,  23., 211., 127.,  11., 132.,  53., 192.,  69.,
        200.,  45., 172., 107., 158., 104.,  51., 130., 124., 102.,  20.,
        216.,  21., 126.,  44.,   9.,  45., 135.,  34.],

## Clara LHS 2/23/2024                          
    # 'stock_red_vol':         [ 82., 165., 105.,  26., 109.,  84.,  32., 214., 114., 139., 214., 15.],
    # 'stock_blue_vol':        [ 39.,  76.,  78., 122., 128., 124., 153.,  37., 125., 100.,  39., 69.],
    # 'stock_yellow_vol':      [177.,  57., 116., 150.,  62.,  91., 114.,  47.,  59.,  59.,  46., 214.] ,                          

## Arthur 2/16/2024
#     'stock_red_vol':         [ 100,   50,   200,  50,  150,  0 ,  150,  25,  75,    25,  75,   100],
#     'stock_blue_vol':        [ 100,  200,  50,   50,  150 , 150 ,  0 ,  75,  25,   175,  200,  70],
#     'stock_yellow_vol':      [ 100,   50,  50,   200,   0 ,  150, 150,  200, 200,  100,  25,   130],
#     'stock_black_vol':       [   0,    0,    0,    0,    0,   0,   0,    0,   0,    0,    0,    0],
                          
## Arthur Rainbow
                           ## red    ora  yel   gre   blu   ind   vio    all 
#    'stock_red_vol':         [85,  95,   0,  0,   0, 95, 152,  50,  0,  0,  0,  0],
#    'stock_blue_vol':        [  0,   0,   0, 95, 180, 95,  38, 50,  0,  0,  0,  0],
#    'stock_yellow_vol':      [ 35,  95, 180, 95,   0,  0,   0, 50,  0,  0,  0,  0],
#    'stock_black_vol':       [ 70,   0,  10,  0,  10,  0,   0, 40,  0,  0,  0,  0],
                          
#     'stock_red_vol':         [170, 150, 150, 140, 130,  0,   0,  0],
#     'stock_blue_vol':        [ 30,   0,   0,   0,  30,  0,   0,  0],
#     'stock_yellow_vol':      [  0,   0,  40,  40,  30,  0,   0,  0],
#     'stock_black_vol':       [  0,  50,   0,  10,   0,  0,   0,  0],
                          
    }
RGB_stocks = pd.DataFrame(data=RGB_stocksandlocations)
RGB_stocks.head()

,mix_loc,stock_red_vol,stock_blue_vol,stock_yellow_vol
0,"Location(point=Point(x=146.88, y=74.24, z=8.52...",40.0,103.0,156.0
1,"Location(point=Point(x=146.88, y=65.24, z=8.52...",130.0,103.0,65.0
2,"Location(point=Point(x=146.88, y=56.24, z=8.52...",67.0,74.0,158.0
3,"Location(point=Point(x=146.88, y=47.24, z=8.52...",3.0,108.0,187.0
4,"Location(point=Point(x=146.88, y=38.24, z=8.52...",43.0,129.0,127.0


In [26]:
red_stock_vol    = pd.Series.to_list(RGB_stocks['stock_red_vol'])
blue_stock_vol   = pd.Series.to_list(RGB_stocks['stock_blue_vol'])
yellow_stock_vol = pd.Series.to_list(RGB_stocks['stock_yellow_vol'])


red_stock_loc   = start_stock['magenta_loc'][0]
blue_stock_loc    = start_stock['cyan_loc'][0]
yellow_stock_loc = start_stock['yellow_loc'][0]

dispense = pd.Series.to_list(RGB_stocks['mix_loc'])

# Run the protocol

In [27]:
## commands
protocol.home()
pip.transfer(red_stock_vol,red_stock_loc ,dispense,blow_out=True,blowout_location="destination well" ,new_tip='once')
#pip.drop_tip()
# move blue 
pip.transfer(blue_stock_vol,blue_stock_loc ,dispense,blow_out=True,blowout_location="destination well" ,new_tip='once') 
# #pip.drop_tip()
# # move yellow
pip.transfer(yellow_stock_vol,yellow_stock_loc,dispense,blow_out=True,blowout_location="destination well",mix_after=(5, 75),new_tip='once') 
   
for line in protocol.commands(): 
    print(line)

#for i in range(1,len(dispense)+1):
    #pip.mix(3, 100, plate[let+str(i)])   # mix 3 times, 100uL, in all plates 
   
for line in protocol.commands(): 
    print(line)

Transferring [40.0, 130.0, 67.0, 3.0, 43.0, 142.0, 118.0, 59.0, 78.0, 38.0, 87.0, 139.0, 135.0, 82.0, 88.0, 76.0, 150.0, 59.0, 203.0, 151.0, 121.0, 114.0, 177.0, 110.0, 140.0, 51.0, 82.0, 73.0, 164.0, 174.0, 109.0, 119.0, 82.0, 96.0, 8.0, 64.0, 125.0, 166.0, 83.0, 60.0, 150.0, 191.0, 148.0, 84.0, 95.0, 105.0, 148.0, 113.0, 123.0, 104.0, 1.0, 89.0, 30.0, 17.0, 128.0, 10.0, 128.0, 41.0, 72.0, 75.0, 117.0, 141.0, 40.0, 57.0, 125.0, 34.0, 25.0, 64.0, 63.0, 217.0, 86.0, 91.0, 156.0, 86.0, 131.0, 79.0, 125.0, 92.0, 119.0, 44.0, 57.0, 19.0, 190.0, 134.0, 129.0, 52.0, 70.0, 240.0, 25.0, 162.0, 21.0, 159.0, 237.0, 120.0, 53.0, 134.0] from A1 of Corning 6 Well Plate 16.8 mL Flat on 1 to A1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2
Picking up tip from A1 of Opentrons OT-2 96 Tip Rack 300 µL on 8
Aspirating 40.0 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on 1 at 92.86 uL/sec
Dispensing 40.0 uL into A1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2 at 92.86 uL/se